## Extract data from gridded dataset Rain for Peru and Ecuador (R4PE; T311_Fernandez-Palomino et al.2022) and Brazilian Daily Weather Gridded Data (BR-DWGD; S208_Xavier et al.2021) https://sites.google.com/site/alexandrecandidoxavierufes/brazilian-daily-weather-gridded-data?authuser=0 
### Created on 2024.04.11
### Last edited 2024.05.14

In [5]:
import pyleoclim as pyleo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
os.chdir('C:/Users/danny/OneDrive - Universidad Politecnica Salesiana/Research/EGU2024/GDD_cave/bin')
print("Current Working Directory:", os.getcwd())



Current Working Directory: C:\Users\danny\OneDrive - Universidad Politecnica Salesiana\Research\EGU2024\GDD_cave\bin


In [6]:
'''
1. R4PE 1981-2015

Three files for each station are created:
a. Daily value from RAIN4PE_daily_0.1d_1981_2015_v1.0.nc. using CDO are save as .txt. 
b. Monthly sum is created as xxx_r4pe_mon.csv
c. Monthly climatology (Jan, Feb, Mar, .... Dec) is created xxx_r4pe_mon_climatology.xls
d. Excel file summary grouping the monthly climatology for all stations and output as combined_climatology.xlsx

'''
import os
import subprocess
import pandas as pd

# Define the list of locations with names, latitudes, and longitudes
locations = [
    {"name": "jumandycave", "lat":  -0.866667, "lon": -77.783333},              #650 masl
    {"name": "ikiam", "lat": -0.950000, "lon": -77.850000},                     #610 masl
    {"name": "dinocave", "lat": -1.425, "lon": -78.040},                        #1200 masl
    
    {"name": "tayuntscave", "lat": -3.022283, "lon": -78.13590},
    {"name": "ecsf", "lat": -3.971667, "lon": -79.079167},
    {"name": "laipuna", "lat": -4.2, "lon": -79.883},
    
    {"name": "porvenircave", "lat": -4.537983, "lon": -79.068583},
    {"name": "shatucacave", "lat": -5.70, "lon": -77.90},
    {"name": "palestinacave", "lat": -5.92, "lon": -77.35},
    
    {"name": "tigrecave", "lat": -5.9406, "lon": -77.308},
    {"name": "huagapocave", "lat": -11.27, "lon": -75.79},

]


# Define the directory paths and filenames
source_directory_path = '/mnt/c/Users/danny/"OneDrive - Universidad Politecnica Salesiana"/Research/EGU2024/GDD_cave/data/r4pe'
#destination_directory_path = 'C:/Users/danny/OneDrive/IKER Speleothems/Dino Spelo/data/r4pe'
destination_directory_path = 'C:/Users/danny/OneDrive - Universidad Politecnica Salesiana/Research/EGU2024/GDD_cave/data/r4pe'
input_filename = "RAIN4PE_daily_0.1d_1981_2015_v1.0.nc"

# Ensure the destination directory exists
os.makedirs(destination_directory_path, exist_ok=True)

climatology_paths = []  # List to store paths of climatology files for later merging

for location in locations:
    name = location["name"]
    lon = location["lon"]
    lat = location["lat"]
    
    # Construct CDO commands and filenames
    output_nc_filename = f"{name}.nc"
    output_txt_filename = f"{name}.txt"
    cleaned_txt_filename = f"{name}_r4pe.txt"
    
    remap_command = f'cd {source_directory_path} && cdo remapnn,lon={lon}_lat={lat} {input_filename} {output_nc_filename}'
    export_command = f'cd {source_directory_path} && cdo outputtab,date,lon,lat,value {output_nc_filename} > {output_txt_filename}'
    
    # Execute CDO commands
    try:
        subprocess.run(["wsl", "bash", "-c", remap_command], check=True, capture_output=True, text=True)
        subprocess.run(["wsl", "bash", "-c", export_command], check=True, capture_output=True, text=True)
        print(f"CDO commands executed successfully for {name}.")
    except subprocess.CalledProcessError as e:
        print(f"Error running CDO command for {name}: {e.stderr}")
    
    # Process and save daily data
    original_file_path = os.path.join(source_directory_path.replace('/mnt/c', 'C:').replace('"', ''), output_txt_filename)
    new_file_path = os.path.join(destination_directory_path, cleaned_txt_filename)
    
    # Read and clean data
    with open(original_file_path, 'r') as file:
        lines = file.readlines()[:-1]
    with open(new_file_path, 'w') as new_file:
        for line in lines:
            new_file.write(line)
    
    # Format and save cleaned data
    df = pd.read_csv(new_file_path, delim_whitespace=True, header=None, skiprows=1, names=['date', 'lon', 'lat', 'value'])
    df['date'] = pd.to_datetime(df['date']).dt.strftime('%d/%m/%Y')
    df.to_csv(new_file_path, sep='\t', index=False, header=True)
    print(f"Processed data and {cleaned_txt_filename} has been created in {destination_directory_path}.")

    # Create monthly time series for daily precipitation
    df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
    df['month_year'] = df['date'].dt.to_period('M')
    monthly_totals = df.groupby('month_year')['value'].sum().reset_index()
    monthly_totals['month_year'] = monthly_totals['month_year'].dt.strftime('%m/%Y')
    
    monthly_file_path = os.path.splitext(new_file_path)[0] + '_mon.csv'
    monthly_totals.to_csv(monthly_file_path, index=False)
    print(f"Monthly time series saved: {monthly_file_path}")

    # Calculate the total monthly precipitation for each month in each year
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    monthly_totals = df.groupby(['year', 'month'])['value'].sum().reset_index()

    # Calculate the climatology by averaging the total monthly precipitation across all years
    monthly_climatology = monthly_totals.groupby('month')['value'].mean().reset_index()
    monthly_climatology.columns = ['Month', 'Average Monthly Precipitation (mm/month)']

    # Save the climatology to CSV
    climatology_file_path = os.path.splitext(monthly_file_path)[0] + '_climatology.csv'
    monthly_climatology.to_csv(climatology_file_path, index=False)
    print(f"Climatology saved: {climatology_file_path}")
    
    climatology_paths.append(climatology_file_path)

# Merge all climatology data into one Excel file
all_climatology_data = {location['name']: pd.read_csv(path).set_index('Month') for location, path in zip(locations, climatology_paths)}
combined_climatology = pd.concat(all_climatology_data.values(), axis=1)
combined_climatology.columns = [location['name'] for location in locations]  # Set column headers as location names
combined_excel_path = os.path.join(destination_directory_path, 'combined_climatology.xlsx')
combined_climatology.to_excel(combined_excel_path)
print(f"All climatology data has been combined into one Excel file: {combined_excel_path}")

Error running CDO command for jumandycave: 


FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/danny/OneDrive - Universidad Politecnica Salesiana/Research/EGU2024/GDD_cave/data/r4pe\\jumandycave.txt'

In [3]:
'''
2. BR-DWGD 1961-2020
'''
import subprocess
import os
import pandas as pd

# Define the list of locations with names, latitudes, and longitudes
locations = [
    {"name": "paraisocave", "lat": -4.07, "lon": -55.45},
    {"name": "lapagrandecave", "lat": -14.4227, "lon": -44.3656},
    {"name": "tocadaboavistacave", "lat": -10.1602, "lon": -40.8605},
    {"name": "curupiracave", "lat": -15.2003, "lon": -56.7839},
    {"name": "tamborilcave", "lat": -16.364, "lon": -46.904}

]

# Define the directory paths and filenames
source_directory_path = '/mnt/c/Users/danny/"OneDrive - Universidad Politecnica Salesiana"/Research/EGU2024/GDD_cave/data/brdwgd'
destination_directory_path = 'C:/Users/danny/OneDrive - Universidad Politecnica Salesiana/Research/EGU2024/GDD_cave/data/brdwgd'
input_filename = "pr_19810101_20001231_BR-DWGD_UFES_UTEXAS_v_3.2.2.nc"


# Ensure the destination directory exists
os.makedirs(destination_directory_path, exist_ok=True)

climatology_paths = []  # List to store paths of climatology files for later merging

for location in locations:
    name = location["name"]
    lon = location["lon"]
    lat = location["lat"]
    
    # Construct CDO commands and filenames
    output_nc_filename = f"{name}.nc"
    output_txt_filename = f"{name}.txt"
    cleaned_txt_filename = f"{name}_r4pe.txt"
    
    remap_command = f'cd {source_directory_path} && cdo remapnn,lon={lon}_lat={lat} {input_filename} {output_nc_filename}'
    export_command = f'cd {source_directory_path} && cdo outputtab,date,lon,lat,value {output_nc_filename} > {output_txt_filename}'
    
    # Execute CDO commands
    try:
        subprocess.run(["wsl", "bash", "-c", remap_command], check=True, capture_output=True, text=True)
        subprocess.run(["wsl", "bash", "-c", export_command], check=True, capture_output=True, text=True)
        print(f"CDO commands executed successfully for {name}.")
    except subprocess.CalledProcessError as e:
        print(f"Error running CDO command for {name}: {e.stderr}")
    
    # Process and save daily data
    original_file_path = os.path.join(source_directory_path.replace('/mnt/c', 'C:').replace('"', ''), output_txt_filename)
    new_file_path = os.path.join(destination_directory_path, cleaned_txt_filename)
    
    # Read and clean data
    with open(original_file_path, 'r') as file:
        lines = file.readlines()[:-1]
    with open(new_file_path, 'w') as new_file:
        for line in lines:
            new_file.write(line)
    
    # Format and save cleaned data
    df = pd.read_csv(new_file_path, delim_whitespace=True, header=None, skiprows=1, names=['date', 'lon', 'lat', 'value'])
    df['date'] = pd.to_datetime(df['date']).dt.strftime('%d/%m/%Y')
    df.to_csv(new_file_path, sep='\t', index=False, header=True)
    print(f"Processed data and {cleaned_txt_filename} has been created in {destination_directory_path}.")

    # Create monthly time series for daily precipitation
    df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
    df['month_year'] = df['date'].dt.to_period('M')
    monthly_totals = df.groupby('month_year')['value'].sum().reset_index()
    monthly_totals['month_year'] = monthly_totals['month_year'].dt.strftime('%m/%Y')
    
    monthly_file_path = os.path.splitext(new_file_path)[0] + '_mon.csv'
    monthly_totals.to_csv(monthly_file_path, index=False)
    print(f"Monthly time series saved: {monthly_file_path}")

    # Calculate the total monthly precipitation for each month in each year
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    monthly_totals = df.groupby(['year', 'month'])['value'].sum().reset_index()

    # Calculate the climatology by averaging the total monthly precipitation across all years
    monthly_climatology = monthly_totals.groupby('month')['value'].mean().reset_index()
    monthly_climatology.columns = ['Month', 'Average Monthly Precipitation (mm/month)']

    # Save the climatology to CSV
    climatology_file_path = os.path.splitext(monthly_file_path)[0] + '_climatology.csv'
    monthly_climatology.to_csv(climatology_file_path, index=False)
    print(f"Climatology saved: {climatology_file_path}")
    
    climatology_paths.append(climatology_file_path)

# Merge all climatology data into one Excel file
all_climatology_data = {location['name']: pd.read_csv(path).set_index('Month') for location, path in zip(locations, climatology_paths)}
combined_climatology = pd.concat(all_climatology_data.values(), axis=1)
combined_climatology.columns = [location['name'] for location in locations]  # Set column headers as location names
combined_excel_path = os.path.join(destination_directory_path, 'combined_climatology.xlsx')
combined_climatology.to_excel(combined_excel_path)
print(f"All climatology data has been combined into one Excel file: {combined_excel_path}")



CDO commands executed successfully for paraisocave.
Processed data and paraisocave_r4pe.txt has been created in C:/Users/danny/OneDrive - Universidad Politecnica Salesiana/Research/EGU2024/GDD_cave/data/brdwgd.
Monthly time series saved: C:/Users/danny/OneDrive - Universidad Politecnica Salesiana/Research/EGU2024/GDD_cave/data/brdwgd\paraisocave_r4pe_mon.csv
Climatology saved: C:/Users/danny/OneDrive - Universidad Politecnica Salesiana/Research/EGU2024/GDD_cave/data/brdwgd\paraisocave_r4pe_mon_climatology.csv
CDO commands executed successfully for lapagrandecave.
Processed data and lapagrandecave_r4pe.txt has been created in C:/Users/danny/OneDrive - Universidad Politecnica Salesiana/Research/EGU2024/GDD_cave/data/brdwgd.
Monthly time series saved: C:/Users/danny/OneDrive - Universidad Politecnica Salesiana/Research/EGU2024/GDD_cave/data/brdwgd\lapagrandecave_r4pe_mon.csv
Climatology saved: C:/Users/danny/OneDrive - Universidad Politecnica Salesiana/Research/EGU2024/GDD_cave/data/brdwg

this is text